# RAG Generation

This stage connects the completed retrieval and reranking pipeline to a local instruction-tuned LLM.

The generation model is `qwen3:8b` served locally through Ollama. The goal is to establish a clean, evidence-grounded end-to-end RAG path before systematic generation evaluation.


## Architecture

`question → PubMedBERT retrieval → top 20 candidates → biomedical reranker → top 5 context → Qwen3-8B → grounded answer`

The retrieval architecture selected in Step 08 is preserved. Generation consumes the ranked evidence produced by retrieval and reranking.


## Why Qwen3-8B

Qwen3-8B is small enough to run locally while providing a meaningful instruction-following generation component.

The initial implementation uses non-thinking mode so that the generation path stays focused on evidence-grounded answering rather than adding unnecessary reasoning latency.


## Local setup

Ollama must be installed and running locally.

```bash
ollama pull qwen3:8b
```

Keep the Ollama server running in a separate terminal while executing the RAG pipeline.


## Implementation

Step 09 adds four descriptive Python modules:

- `build_pubmedbert_vector_store.py` — builds the separate PubMedBERT Qdrant collection.
- `reranking.py` — reusable biomedical cross-encoder reranker.
- `rag_generation.py` — prompt construction and Ollama generation.
- `rag_pipeline.py` — executable end-to-end RAG pipeline.

The original MiniLM vector store and completed retrieval-stage scripts are not replaced.


## Run order

From the project root:

```bash
python src/build_pubmedbert_vector_store.py
python src/rag_pipeline.py
```

The first command builds the separate `data/qdrant_pubmedbert` collection named `pubmedqa_pubmedbert`.

The second command executes:

`question → PubMedBERT top 20 → biomedical reranker → top 5 context → Qwen3-8B`


## Observed result

The end-to-end pipeline completed successfully on the test question:

> Do mitochondria play a role in the pathogenesis of non-alcoholic fatty liver disease?

The pipeline retrieved 20 candidates, reranked them, passed the top 5 contexts to Qwen3-8B, and produced a grounded response.

The retrieved contexts were not actually relevant to the specific mitochondrial-mechanism question. The top result concerned distinguishing NASH from alcoholic steatohepatitis, and the remaining retrieved passages were similarly indirect.

The generation model therefore correctly abstained rather than supplying an answer from outside knowledge.


## Findings

The Step 09 smoke test demonstrates that the complete RAG path is operational and that the generation prompt enforces evidence grounding.

The observed example also exposes an important system-level failure mode: a fluent language model cannot compensate for inadequate retrieval evidence. In this case, generation behaved appropriately by stating that the retrieved context was insufficient.

This is only a single qualitative test and is not evidence of overall generation quality. The next stage will evaluate the pipeline systematically across a reproducible evaluation set, measuring answer quality and grounding and analyzing failure cases.


## Scope boundary

This stage establishes generation only. Agents, tool calling, FastAPI, Docker, and production serving are intentionally deferred until the end-to-end RAG system has been evaluated.
